<a href="https://colab.research.google.com/github/shaheem1771/Chatbot/blob/main/Campus_assistant_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq

In [ ]:
from groq import Groq

client = Groq(api_key="api key")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response.choices[0].message.content)

Hello. How can I assist you today?


In [ ]:
knowledge = """
LBS College of Engineering, Kasaragod (LBSCEK)

Established: 1993

Location:
Povval, Muliyar P.O., Kasaragod, Kerala - 671542

Affiliation:
APJ Abdul Kalam Technological University (KTU)

Vision:
To become a paragon institution for pursuance of Education and Research in Engineering and Technology.

Departments:
- Computer Science and Engineering
- Computer Science and Engineering (AI & DS)
- Computer Science and Business Systems
- Electronics and Communication Engineering
- Electrical and Electronics Engineering
- Mechanical Engineering
- Civil Engineering
- Information Technology

Facilities:
- Central Library
- Digital Library
- Hostel Facility
- Bus Service
- ATM Facility
- Fab Lab
- WiFi Campus
- Auditorium
- Sports Facilities

Student Activities:
- NSS
- IEEE
- IEDC
- CGPU
- College Union
- PTA

Admissions:
- KEAM
- NRI Admission
- Lateral Entry
- Fee Waiver Scheme

Campus Area:
52 Acres

Contact:
04994 250290
04994 250555

Website:
https://lbscek.ac.in
"""

In [ ]:
question = "What facilities are available in LBSCEK?"

prompt = f"""
You are LBSCEK Student Assistant.

Answer ONLY using the information below.
If the answer is not available, say:
'Sorry, I don't have that information.'

{knowledge}

Question:
{question}
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

The facilities available in LBSCEK are:

1. Central Library
2. Digital Library
3. Hostel Facility
4. Bus Service
5. ATM Facility
6. Fab Lab
7. WiFi Campus
8. Auditorium
9. Sports Facilities


In [ ]:
question = "How can I get admission to LBSCEK?"

prompt = f"""
You are LBSCEK Student Assistant.

Answer only using the information provided below.

If the answer is not available in the information, say:
Sorry, I don't have that information.

Knowledge Base:

{knowledge}

Question:
{question}
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print("\nAnswer:\n")
print(response.choices[0].message.content)


Answer:

You can get admission to LBSCEK through KEAM, NRI Admission, Lateral Entry, or Fee Waiver Scheme.


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
print(gr.__version__)

5.50.0


In [ ]:
# @title
 import gradio as gr

def lbs_chat(message, history):

    prompt = f"""
You are LBS Connect.

Answer ONLY using the information below.

If the answer is not available, reply exactly:
Sorry, I don't have that information.

{knowledge}

Question:
{message}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate"
    ),
    css="""
    footer{
        display:none!important;
    }

    body{
        background:#020817!important;
    }

    .gradio-container{
        max-width:900px!important;
        margin:auto!important;
    }

    h1{
        text-align:center!important;
        font-size:56px!important;
        font-weight:800!important;
        color:white!important;
        margin-bottom:0!important;
    }

    .subtitle{
        text-align:center;
        color:#94a3b8;
        font-size:18px;
        margin-bottom:20px;
    }

    .chatbox{
        border-radius:20px!important;
        border:1px solid rgba(255,255,255,.08)!important;
        background:#08122e!important;
    }

    textarea{
        border-radius:14px!important;
    }

    button{
        border-radius:14px!important;
    }

    .faq-card{
        width:100%!important;
        height:58px!important;
        margin-bottom:12px!important;
        font-size:15px!important;
        font-weight:600!important;
        border-radius:14px!important;
    }
    """
) as demo:

    gr.Markdown("""
    # LBS Connect

    <div class="subtitle">
    College Information Portal
    </div>
    """)

    chatbot = gr.Chatbot(
        [],
        height=180,
        type="messages",
        elem_classes="chatbox"
    )

    faq_group = gr.Group(visible=True)

    with faq_group:

        q1 = gr.Button(
            "What courses are offered at LBSCEK?",
            elem_classes="faq-card"
        )

        q2 = gr.Button(
            "What are the admission options at LBSCEK?",
            elem_classes="faq-card"
        )

        q3 = gr.Button(
            "What facilities are available on campus?",
            elem_classes="faq-card"
        )

    with gr.Row():

        msg = gr.Textbox(
            placeholder="Ask anything...",
            show_label=False,
            container=False,
            scale=8
        )

        send = gr.Button(
            "➜",
            variant="primary",
            scale=2
        )

    def respond(message, history):

        answer = lbs_chat(message, history)

        history.append(
            {
                "role": "user",
                "content": message
            }
        )

        history.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        return (
            "",
            history,
            gr.update(visible=False),
            gr.update(height=420)
        )

    send.click(
        respond,
        [msg, chatbot],
        [msg, chatbot, faq_group, chatbot]
    )

    msg.submit(
        respond,
        [msg, chatbot],
        [msg, chatbot, faq_group, chatbot]
    )

    def faq_click(question, history):

        answer = lbs_chat(question, history)

        history.append(
            {
                "role": "user",
                "content": question
            }
        )

        history.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        return (
            history,
            gr.update(visible=False),
            gr.update(height=420)
        )

    q1.click(
        lambda h: faq_click(
            "What courses are offered at LBSCEK?",
            h
        ),
        inputs=[chatbot],
        outputs=[chatbot, faq_group, chatbot]
    )

    q2.click(
        lambda h: faq_click(
            "What are the admission options at LBSCEK?",
            h
        ),
        inputs=[chatbot],
        outputs=[chatbot, faq_group, chatbot]
    )

    q3.click(
        lambda h: faq_click(
            "What facilities are available on campus?",
            h
        ),
        inputs=[chatbot],
        outputs=[chatbot, faq_group, chatbot]
    )

demo.launch(share=True)

/tmp/ipykernel_4478/722891700.py:32: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_4478/722891700.py:32: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_4478/722891700.py:99: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0c2f593584dae4fe80.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
